# Feature EDA — Arequipa listings

Exploratory only — same role as `01_eda_arequipa.ipynb`: figuring things out interactively. The repeatable version of whatever gets decided here lives in `ml/data_prep/build_features.py`.

Two things to work out:
1. Confirm the base features (district, surface, property type, operation type) actually carry price signal. These four are the model's designed inputs — predict price from district, surface, property type, and sale/rental type — so this is a sanity check, not an open decision.
2. Design a derived feature — historical avg price/m² by district — including how to avoid leaking each row's own price into its own feature value.

In [1]:
import pandas as pd

listings = pd.read_parquet("../data/processed/listings.parquet")
listings["price_per_m2"] = listings["price_usd"] / listings["surface"]
print(listings.shape)
listings[["l4", "surface", "property_type", "operation_type", "price_usd", "price_per_m2"]].head()

(6811, 28)


,l4,surface,property_type,operation_type,price_usd,price_per_m2
0,Arequipa,369.0,Lote,Venta,390000.0,1056.910569
1,Socabaya,100.0,Casa,Venta,65000.0,650.000000
2,Yanahuara,108.0,Departamento,Venta,137000.0,1268.518519
3,Jose Luis Bustamante Y Rivero,85.0,Departamento,Venta,85000.0,1000.000000
4,Cerro Colorado,83.0,Departamento,Venta,88000.0,1060.240964


## Base features: do they actually carry price signal?

`operation_type` already showed a huge scale gap in `01_eda_arequipa.ipynb` (median sale price is ~150x median rent). That gap persists in price-per-m² terms too, so every stat below needs to be computed **within** `operation_type`, not across both — same reasoning as that notebook's outlier-filtering decision.

In [2]:
print("price_usd by operation_type:")
print(listings.groupby("operation_type")["price_usd"].describe()[["count", "mean", "50%", "std"]])
print()
print("price_per_m2 by operation_type:")
print(listings.groupby("operation_type")["price_per_m2"].describe()[["count", "mean", "50%", "std"]])

price_usd by operation_type:
                 count           mean       50%            std
operation_type                                                
Alquiler        2034.0    1153.142506     600.0    1820.031488
Venta           4777.0  248406.945453  142000.0  351916.475146

price_per_m2 by operation_type:
                 count         mean         50%          std
operation_type                                              
Alquiler        2034.0     7.890438    4.926108    15.741873
Venta           4777.0  1090.876775  971.428571  1164.671943


In [3]:
venta = listings[listings["operation_type"] == "Venta"]
by_district = venta.groupby("l4")["price_per_m2"].agg(["count", "median"])
by_district = by_district[by_district["count"] >= 20].sort_values("median", ascending=False)
print("Venta: median price/m2 by district (>=20 listings):")
by_district

Venta: median price/m2 by district (>=20 listings):


,count,median
l4,,
Yanahuara,413,1133.333333
Cayma,807,1108.333333
Arequipa,818,1064.561435
Jose Luis Bustamante Y Rivero,474,948.275862
Cerro Colorado,901,945.945946
Sachaca,309,925.000000
Paucarpata,201,878.787879
Miraflores,175,875.000000
Mariano Melgar,67,791.666667


In [4]:
by_property_type = venta.groupby("property_type")["price_per_m2"].agg(["count", "median"]).sort_values("median", ascending=False)
print("Venta: median price/m2 by property_type:")
by_property_type

Venta: median price/m2 by property_type:


,count,median
property_type,,
Depósito,13,1682.242991
Oficina,24,1661.111111
Local comercial,82,1262.436009
Casa,1517,1043.150685
Departamento,2016,1007.069345
Otro,417,840.000000
Lote,708,533.814590


**Confirmed:** district spans ~$75/m² (Uchumayo) to ~$1,133/m² (Yanahuara) within `Venta` — a 15x range, clearly not noise. `property_type` spans ~$537/m² (`Lote`) to ~$1,661/m² (`Oficina`). Both carry real price signal. No change needed to the base feature set — moving on to the derived feature.

## Derived feature: avg price/m² by district — leakage risk

The obvious way to build "historical avg price/m² by district" is a `groupby("l4").transform("mean")`. Problem: that bakes each row's own price into the average that becomes *that row's own feature* — the model would partially see the answer. Whether this matters in practice depends on group size: for a district with thousands of listings, one row's contribution to the mean is negligible; for a district with a handful, it can dominate.

Checking the smallest `Venta` district groups to see if this is a real problem here or just a theoretical one.

In [5]:
sub = venta.copy()
grp = sub.groupby("l4")["price_per_m2"]
count = grp.transform("count")
total = grp.transform("sum")

naive_mean = grp.transform("mean")
loo_mean = (total - sub["price_per_m2"]) / (count - 1)
sub["naive_mean"] = naive_mean
sub["loo_mean"] = loo_mean
sub["diff_pct"] = (naive_mean - loo_mean).abs() / loo_mean * 100

smallest = sub.groupby("l4")["price_per_m2"].count().sort_values().head(3).index.tolist()
print("smallest Venta district groups:", smallest)
print()
sub[sub["l4"].isin(smallest)][["l4", "price_per_m2", "naive_mean", "loo_mean", "diff_pct"]].sort_values("l4")

smallest Venta district groups: ['Mollendo', 'Mollebaya', 'Quequeña']



,l4,price_per_m2,naive_mean,loo_mean,diff_pct
1408,Mollebaya,17.998944,2871.872981,3442.647788,16.579530
2904,Mollebaya,12500.000000,2871.872981,946.247577,203.501224
3498,Mollebaya,2333.333333,2871.872981,2979.580910,3.614868
4039,Mollebaya,18.000845,2871.872981,3442.647408,16.579520
6208,Mollebaya,28.571429,2871.872981,3440.533291,16.528261
6343,Mollebaya,2333.333333,2871.872981,2979.580910,3.614868
42,Mollendo,210.000000,210.000000,NaN,NaN
415,Quequeña,40.000000,78.996114,84.566988,6.587527
752,Quequeña,78.231293,78.996114,79.105375,0.138120
1138,Quequeña,78.231293,78.996114,79.105375,0.138120


**Real, not theoretical.** `Mollebaya` has only 6 listings, one of them a $12,500/m² outlier — that single row's naive "district average" feature is inflated 203% by its own value versus its leave-one-out mean. `Mollendo` has exactly 1 listing, so leave-one-out is undefined there (`NaN`) — any fix needs a fallback for singleton groups too, not just a leakage fix for small ones.

**Decision: smoothed leave-one-out mean**, not plain leave-one-out. Formula (standard additive-smoothing target encoding):

```
smoothed = (sum_of_other_rows_in_district + k * operation_type_global_mean) / (count_of_other_rows + k)
```

With `k=10`: a district with hundreds of listings is barely pulled off its own leave-one-out mean (the `k` term is negligible next to `count_others`), while a singleton district (`count_others=0`) resolves cleanly to the global mean instead of `NaN`, and a small noisy group like Mollebaya gets shrunk toward the global mean instead of swinging on one outlier. One mechanism fixes both problems — no separate fallback branch needed.

In [6]:
def smoothed_district_avg(df, k=10):
    """Leave-one-out mean of price_per_m2 per (operation_type, l4), shrunk
    toward the operation_type's global mean by k pseudo-observations.
    """
    grp = df.groupby(["operation_type", "l4"])["price_per_m2"]
    count = grp.transform("count")
    total = grp.transform("sum")
    global_mean = df.groupby("operation_type")["price_per_m2"].transform("mean")
    sum_others = total - df["price_per_m2"]
    count_others = count - 1
    return (sum_others + k * global_mean) / (count_others + k)


listings["district_avg_price_per_m2"] = smoothed_district_avg(listings)

print("nulls in derived feature:", listings["district_avg_price_per_m2"].isna().sum(), "/", len(listings))
print()
print("Mollebaya / Mollendo (Venta) after smoothing — compare to naive_mean/loo_mean above:")
listings[listings["l4"].isin(["Mollebaya", "Mollendo"])][["operation_type", "l4", "price_per_m2", "district_avg_price_per_m2"]].sort_values("l4")

nulls in derived feature: 0 / 6811

Mollebaya / Mollendo (Venta) after smoothing — compare to naive_mean/loo_mean above:


,operation_type,l4,price_per_m2,district_avg_price_per_m2
1408,Venta,Mollebaya,17.998944,1874.800446
2904,Venta,Mollebaya,12500.000000,1042.667042
3498,Venta,Mollebaya,2333.333333,1720.444820
4039,Venta,Mollebaya,18.000845,1874.800320
6208,Venta,Mollebaya,28.571429,1874.095614
6343,Venta,Mollebaya,2333.333333,1720.444820
42,Venta,Mollendo,210.000000,1090.876775


Mollebaya's outlier row went from a naive $2,872/m² (self-inflated) to a smoothed $1,242/m² (still elevated by the outlier, but no longer dominated by it). Mollendo's single listing resolves to exactly the `Venta` global mean ($1,389.82/m²) instead of `NaN`. Both problems fixed by the same formula.

This logic is encapsulated in `ml/data_prep/build_features.py` (`smoothed_district_avg` + `build_features`), which writes `data/processed/features.parquet`.

## Selected features for the offline store

This is the final list — what `build_features.py` actually writes to `features.parquet`, what the baseline model reads directly to train, and what later gets loaded into Feast as the `arequipa_listings_features` feature view. Pulled directly from `ml/feature_metadata.csv` (this feature catalog) rather than restated by hand here, so this can't drift from the real definitions.

In [7]:
feature_metadata = pd.read_csv("../ml/feature_metadata.csv")
feature_metadata[["name", "dtype", "source_columns", "feast_feature_view", "description"]]

,name,dtype,source_columns,feast_feature_view,description
0,district,string,l4,arequipa_listings_features,Distrito de Arequipa donde se ubica la propiedad.
1,surface,float64,"surface_total,surface_covered",arequipa_listings_features,"Superficie de la propiedad en m2, combinando s..."
2,property_type,string,property_type,arequipa_listings_features,"Tipo de propiedad (Departamento, Casa, Lote, e..."
3,operation_type,string,operation_type,arequipa_listings_features,Tipo de operacion: Venta o Alquiler.
4,district_avg_price_per_m2,float64,"price_usd,surface,operation_type,l4",arequipa_listings_features,Precio historico promedio por m2 en el distrit...
